# D1.8 · Threat intel sub-lane

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

---

**Risk.** Unsourced confidence in synthesis loops.

**Control.** Provenance discipline; refuse claims without a source.

**This lab.** Build a synthesis loop that must cite or abstain.

| | |
|---|---|
| Open-source tooling | MISP, OpenCTI |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.8"))

Threat intel for agentic systems: the only useful test of a feed is how many detections came out of it.

In [ ]:
from cybercommons import soc
import time

feed = [
    soc.Indicator("collect.example.com", "host", "vendor-a", 0.95),
    soc.Indicator("169.254.169.254", "host", "internal", 0.99),
    soc.Indicator("a1b2c3d4", "hash", "vendor-b", 0.72),
    soc.Indicator("adversaries are increasingly using agents", "technique", "blog", 0.40),
    soc.Indicator("pastebin.example", "host", "vendor-a", 0.55),
]
rules = soc.intel_to_rules(feed)
print(f"{len(feed)} indicators → {len(rules)} deployable rules\n")
for r in rules:
    print(f"  [{r.severity}] {r.name}")

print("\ndropped:")
for i in feed:
    if not i.actionable():
        print(f"  {i.value[:44]:46s} kind={i.kind} conf={i.confidence}")

The dropped rows are the point. A narrative about adversary trends is not intelligence you can operate; a low-confidence host would cost more in false positives than it buys.

In [ ]:
now = time.time()
events = [soc.Event(now, "patch-agent", "http_get", "https://collect.example.com/x")]
for a in soc.run_rules(events, rules):
    print(f"[{a.severity}] {a.rule} → {a.response}")

### Expect

Three of five indicators convert to rules; the technique narrative and the low-confidence host are dropped with reasons. The exfiltration host then fires a high-severity alert with a response.

### Your turn

Compute the conversion rate for your actual intel spend: indicators received versus rules deployed versus alerts actioned. The third number is usually the surprising one.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*